In [0]:
# Install required libraries untuk extract text dari PDF dan DOCX
%pip install PyPDF2 python-docx scikit-learn --quiet
dbutils.library.restartPython()

In [0]:
from PyPDF2 import PdfReader
import os

def extract_text_from_pdf(file_path):
    """
    Extract text dari file PDF
    
    Args:
        file_path: Path ke file PDF (bisa /Volumes atau /dbfs)
    
    Returns:
        String berisi semua text dari PDF
    """
    try:
        reader = PdfReader(file_path)
        text = ""
        
        for page_num, page in enumerate(reader.pages, 1):
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
        
        print(f"✅ Berhasil extract {len(reader.pages)} halaman")
        print(f"📊 Total karakter: {len(text):,}")
        return text.strip()
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Test function (uncomment jika mau test dengan file dummy)
# Contoh: text = extract_text_from_pdf("/Volumes/catalog/schema/volume/document.pdf")

In [0]:
from docx import Document

def extract_text_from_docx(file_path):
    """
    Extract text dari file DOCX (Word)
    
    Args:
        file_path: Path ke file DOCX
    
    Returns:
        String berisi semua text dari DOCX
    """
    try:
        doc = Document(file_path)
        text = ""
        
        # Extract dari paragraphs
        for para in doc.paragraphs:
            if para.text.strip():
                text += para.text + "\n"
        
        # Extract dari tables
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    if cell.text.strip():
                        text += cell.text + " "
            text += "\n"
        
        print(f"✅ Berhasil extract {len(doc.paragraphs)} paragraphs dan {len(doc.tables)} tables")
        print(f"📊 Total karakter: {len(text):,}")
        return text.strip()
    
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

# Test function
# Contoh: text = extract_text_from_docx("/Volumes/catalog/schema/volume/document.docx")

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def calculate_similarity(text1, text2, show_details=True):
    """
    Hitung similarity antara 2 dokumen menggunakan TF-IDF + Cosine Similarity
    
    Args:
        text1: Text dari dokumen pertama
        text2: Text dari dokumen kedua
        show_details: Tampilkan detail analisis (default: True)
    
    Returns:
        Dictionary berisi similarity score dan detail
    """
    if not text1 or not text2:
        return {"error": "Text tidak boleh kosong"}
    
    # Vectorize text menggunakan TF-IDF
    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words='english',  # Filter common words
        max_features=5000  # Max 5000 most important words
    )
    
    try:
        tfidf_matrix = vectorizer.fit_transform([text1, text2])
        
        # Hitung cosine similarity
        similarity_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        
        # Get vocabulary info
        feature_names = vectorizer.get_feature_names_out()
        
        # Get top words from each document
        doc1_vector = tfidf_matrix[0].toarray()[0]
        doc2_vector = tfidf_matrix[1].toarray()[0]
        
        # Top 10 keywords from doc1
        top_indices_doc1 = doc1_vector.argsort()[-10:][::-1]
        top_keywords_doc1 = [feature_names[i] for i in top_indices_doc1 if doc1_vector[i] > 0]
        
        # Top 10 keywords from doc2
        top_indices_doc2 = doc2_vector.argsort()[-10:][::-1]
        top_keywords_doc2 = [feature_names[i] for i in top_indices_doc2 if doc2_vector[i] > 0]
        
        result = {
            "similarity_score": similarity_score,
            "similarity_percentage": round(similarity_score * 100, 2),
            "total_features": len(feature_names),
            "doc1_length": len(text1),
            "doc2_length": len(text2),
            "top_keywords_doc1": top_keywords_doc1,
            "top_keywords_doc2": top_keywords_doc2
        }
        
        if show_details:
            print("="*60)
            print("📊 HASIL ANALISIS SIMILARITY")
            print("="*60)
            print(f"\n🎯 Similarity Score: {result['similarity_percentage']}%")
            print(f"\n📏 Interpretasi:")
            if similarity_score >= 0.8:
                print("   ✅ Sangat Mirip (>80%)")
            elif similarity_score >= 0.6:
                print("   ⚠️  Cukup Mirip (60-80%)")
            elif similarity_score >= 0.4:
                print("   ⚡ Agak Mirip (40-60%)")
            else:
                print("   ❌ Tidak Mirip (<40%)")
            
            print(f"\n📝 Detail:")
            print(f"   - Dokumen 1: {result['doc1_length']:,} karakter")
            print(f"   - Dokumen 2: {result['doc2_length']:,} karakter")
            print(f"   - Total fitur TF-IDF: {result['total_features']:,}")
            
            print(f"\n🔑 Top Keywords Dokumen 1: {', '.join(top_keywords_doc1[:5])}")
            print(f"🔑 Top Keywords Dokumen 2: {', '.join(top_keywords_doc2[:5])}")
            print("="*60)
        
        return result
    
    except Exception as e:
        return {"error": str(e)}

In [0]:
# ==========================================
# 📖 CARA PENGGUNAAN
# ==========================================

"""
STEP 1: Upload dokumen ke DBFS atau Volumes
-----------------------------------------
Upload 2 file (PDF atau DOCX) yang ingin dibandingkan ke:
- Volumes: /Volumes/catalog/schema/volume/
- DBFS: /dbfs/FileStore/

Contoh upload via UI:
1. Klik 'Data' di sidebar kiri
2. Pilih Volume atau create folder di FileStore
3. Upload file PDF/DOCX


STEP 2: Jalankan comparison
--------------------------
"""

# CONTOH 1: Compare 2 PDF Files
print("\n🔍 Contoh 1: Compare 2 PDF Files\n")

# Ganti path dengan lokasi file Anda
file1_path = "/Volumes/catalog/schema/volume/document1.pdf"  # ⬅️ EDIT INI
file2_path = "/Volumes/catalog/schema/volume/document2.pdf"  # ⬅️ EDIT INI

# Uncomment 3 baris di bawah untuk menjalankan
# text1 = extract_text_from_pdf(file1_path)
# text2 = extract_text_from_pdf(file2_path)
# result = calculate_similarity(text1, text2)


# CONTOH 2: Compare 2 DOCX Files
print("\n🔍 Contoh 2: Compare 2 DOCX Files\n")

# Ganti path dengan lokasi file Anda
file1_path = "/Volumes/catalog/schema/volume/document1.docx"  # ⬅️ EDIT INI
file2_path = "/Volumes/catalog/schema/volume/document2.docx"  # ⬅️ EDIT INI

# Uncomment 3 baris di bawah untuk menjalankan
# text1 = extract_text_from_docx(file1_path)
# text2 = extract_text_from_docx(file2_path)
# result = calculate_similarity(text1, text2)


# CONTOH 3: Compare PDF vs DOCX
print("\n🔍 Contoh 3: Compare PDF vs DOCX\n")

pdf_path = "/Volumes/catalog/schema/volume/document.pdf"    # ⬅️ EDIT INI
docx_path = "/Volumes/catalog/schema/volume/document.docx"  # ⬅️ EDIT INI

# Uncomment 3 baris di bawah untuk menjalankan
# text1 = extract_text_from_pdf(pdf_path)
# text2 = extract_text_from_docx(docx_path)
# result = calculate_similarity(text1, text2)


# CONTOH 4: Test dengan sample text (tanpa file)
print("\n🔍 Contoh 4: Test dengan Sample Text\n")
print("Uncomment code di bawah untuk test langsung:\n")

# Sample documents
sample_text1 = """
Databricks adalah platform data lakehouse yang menggabungkan data warehousing dan data lake.
Platform ini mendukung Apache Spark, machine learning, dan analytics dalam satu unified platform.
Databricks menyediakan collaborative workspace untuk data engineers dan data scientists.
"""

sample_text2 = """
Databricks merupakan unified analytics platform yang mengintegrasikan data lake dan warehouse.
Mendukung distributed computing dengan Spark, serta machine learning workflows.
Platform kolaboratif untuk tim data engineering dan data science.
"""

# Uncomment baris di bawah untuk test
# result = calculate_similarity(sample_text1, sample_text2)


print("\n" + "="*60)
print("✅ Setup selesai! Ikuti instruksi di atas untuk mulai compare dokumen.")
print("="*60)